# 06 Showcase data

This notebook gathers the results from notebooks 01 to 05 into `docs/data.js`, the only data file the website reads. Numbers on the site therefore come from the same tables as the Insight Report.

In [ ]:
%run 00_utils.ipynb

import json
import pandas as pd

COMMUNITY_NAMES = {"C1": "HIV generics core", "C2": "TB generics", "C3": "Malaria, injectables and originators",
                   "C4": "TB patient kits", "C5": "Single-originator ARVs", "C6": "Arteether", "C7": "Pyronaridine-artesunate"}

## Product map

In [ ]:
# medicines and their suppliers in the main network spec
nodes = pd.read_csv(NETWORK / "product_nodes_analyzed_min2.csv", index_col=0)
main_links = pd.read_csv(NETWORK / "bipartite_edges_min2.csv")
main_proximity = pd.read_csv(NETWORK / "product_edges_min2.csv")

# firms are listed from largest to smallest spend so the map can name the main suppliers
suppliers = main_links.sort_values("spend_usd", ascending=False).groupby("product")["firm"].apply(list)

map_nodes = [{"id": name, "area": row.area, "community": row.community, "spend": round(row.spend_usd / 1e6, 1),
              "firms": suppliers.get(name, [])} for name, row in nodes.iterrows()]

# only strong ties are drawn so the map stays readable
strong_ties = main_proximity[main_proximity.phi >= 0.5]
map_edges = [[source, target] for source, target in zip(strong_ties.source, strong_ties.target)]

## Candidate suppliers

In [ ]:
# use the same links as the candidate model: every firm-product link that began by 2025
links_2025 = pd.read_csv(NETWORK / "bipartite_edges_min1.csv").query("first_year < 2026")
proximity = project_products(links_2025)
both_directions = pd.concat([proximity.rename(columns={"source": "product", "target": "neighbor"}),
                             proximity.rename(columns={"target": "product", "source": "neighbor"})])
firm_products = links_2025.groupby("firm")["product"].apply(set)


def closest_medicines(product, n_closest=5):
    """Return the medicines that share the most suppliers with this one."""
    rows = both_directions[both_directions["product"] == product].sort_values(["phi", "neighbor"], ascending=[False, True])
    return rows.head(n_closest)["neighbor"].tolist()


candidate_table = pd.read_csv(TABLES / "t12_candidate_suppliers.csv")
medicines = []
for product, group in candidate_table.groupby("product", sort=False):
    group = group.sort_values("rank")
    neighbors = closest_medicines(product)
    first = group.iloc[0]
    medicines.append({
        "name": product, "area": first.area, "spend": round(float(first.spend_2021_25_usd_m), 1),
        "current": first.current_suppliers.split("; "), "neighbors": neighbors,
        "candidates": [{"firm": row.firm, "country": None if pd.isna(row.hq_country) else row.hq_country,
                        "p": round(float(row.p_entry_3y), 4), "density": round(float(row.density), 3),
                        "who": bool(row.who_pq_listed),
                        "supplies": [neighbor in firm_products.get(row.firm, set()) for neighbor in neighbors]}
                       for row in group.itertuples()]})

## Evidence and headline numbers

In [ ]:
network_summary = pd.read_csv(TABLES / "t8_network_summary.csv").set_index("metric")["value"]
null_draws = pd.read_csv(NETWORK / "null_model_min2.csv")
density_rates = pd.read_csv(TABLES / "t11b_entry_by_density.csv")
prediction = pd.read_csv(TABLES / "t11_entry_prediction.csv").set_index("model")
entry_panel = pd.read_csv(PROCESSED / "entry_panel.csv").query("firm_active_recent and prod_bought_recent")
test_rows = entry_panel[entry_panel.cutoff >= 2020]

# thin products use the same definition as 04_entry: at most two manufacturers in 2021-2025
pqr = load_clean("pqr_medicines")
recent_suppliers = pqr[pqr.year.between(2021, 2025)].groupby("product")["firm"].nunique()

stats = {
    "products": len(nodes), "communities": int(network_summary["communities"]), "modularity": float(network_summary["modularity_Q"]),
    "null_modularity": float(network_summary["null_Q_mean"]), "nmi": float(network_summary["NMI_area"]),
    "pairs": len(entry_panel), "entries": int(entry_panel.entry.sum()), "test_entries": int(test_rows.entry.sum()),
    "recall_controls": float(prediction.loc["Controls logit", "recall_top5%"]),
    "recall_density": float(prediction.loc["Density logit", "recall_top5%"]),
    "thin": int((recent_suppliers <= 2).sum()), "scored": len(medicines),
}

## Write the website data

In [ ]:
site_data = {"stats": stats, "communities": COMMUNITY_NAMES, "nodes": map_nodes, "edges": map_edges, "medicines": medicines,
             "null": null_draws.Q.round(4).tolist(), "density": density_rates[["density_bin", "pairs", "entry_rate_%"]].values.tolist()}

docs = ROOT / "docs"
docs.mkdir(exist_ok=True)
(docs / "data.js").write_text("window.SITE_DATA = " + json.dumps(site_data, ensure_ascii=False) + ";\n")
print(stats)